# HEART / Ugi-4CR | Priority 1: Evidence-based Virtual Library Validation

**Research question:** Are the approximately 15.18 million records in our existing Ugi-4CR virtual molecular library of sufficient quality to support **LUMI-style continual pretraining**?

> **This is one reproducible, read-only Jupyter Notebook.** It contains the dataset checks, summary statistics, charts, real LMDB sampling, molecular identity audit, and an evidence-based readiness decision. **It does not modify the existing LMDB, Stage-06 sidecars, split assignments, or checkpoints, and it does not submit training jobs.**

### Validation layers (do not conflate them)

1. **Source inventory:** inspect the manifest, whole-shard train/validation split, historical Stage-05 quality control, and the Stage-06 final dataset view / readiness receipt.
2. **Full 3D index census:** inspect every shard's 3D-success index and verify value ranges, record counts, and the distribution of valid conformers.
3. **Full molecular audit:** read every `records.csv.gz`; verify `key`, RDKit-parsable SMILES, canonical molecular identity, `n_3d_success` and agreement with the Stage-06 index; quantify duplicates, exact train/validation molecular leakage, and chemical properties.
4. **Real LMDB payload probe:** sample actual pickle records and 3D coordinates; verify `atoms`, the ten coordinate slots, the successfully generated 3D prefix, chemical identity, and associated metadata.
5. **Visual evidence and readiness decision:** report auditable counts, plots, real failure examples, and provenance; distinguish `GO_DATA`, `REVIEW`, `HOLD`, and `INCOMPLETE`. A **production loader dry-run** is still separately required to authorize model training.

**Data sources and constraints:** This notebook follows the paths and Stage-06 contract documented in `HEART_Ugi_LNPDB_Audit_Decision(2).ipynb`: `manifest.json`, `splits.json`, `quality_report.json`, `library_3d_ready.json`, `final_dataset_view.json`, `shard-*/records.csv.gz`, and Stage-06 index files. Each index byte records the valid 3D prefix length (1–10) for one record. The source LMDB may retain fallback slots. A historical Stage-05 **FAIL is not automatically relabeled PASS**.

**Scope:** This notebook does **not** evaluate LNPDB. Overlap analysis and LNPDB integration are Priority 2.

**Execution environment:** Run in an authorized Slurm CPU allocation or Jupyter compute session with access to `/project/yli4/ypan/`. The full 15M+ canonicalization / SQLite audit requires substantial CPU, memory, and scratch storage. **Do not run the long full scan on a shared login node.** The first run creates a resumable SQLite cache; subsequent runs validate manifest, split, and view fingerprints before safely resuming.


## 0 | Paths, dependencies, and run switches

The defaults target our HPC environment. Override them with the `HEART_UGI_ROOT`, `HEART_UGI_VIEW`, `HEART_AUDIT_OUT`, and `HEART_REPO` environment variables if necessary. Setting `RUN_FULL_AUDIT=True` **reads the full Ugi metadata collection** but never modifies source files.


In [ ]:
from pathlib import Path
from collections import Counter, defaultdict
from datetime import datetime, timezone
from IPython.display import display, Markdown
import os, sys, json, csv, gzip, pickle, sqlite3, hashlib, random, time, traceback, subprocess
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from rdkit import Chem, RDLogger
RDLogger.DisableLog('rdApp.*')

UGI = Path(os.getenv('HEART_UGI_ROOT','/project/yli4/ypan/lipid_library/lmdb')).expanduser()
VIEW = Path(os.getenv('HEART_UGI_VIEW','/project/yli4/ypan/audits/ugi_3d_final_view')).expanduser()
REPO = Path(os.getenv('HEART_REPO','/home/ypan/heart_liver/model_train/heart-molecular-pretraining')).expanduser()
GENERAL_CKPT = Path(os.getenv('HEART_GENERAL_CKPT','/project/yli4/ypan/runs/h200/full/heart-h200-b448-v1/checkpoint_best.pt')).expanduser()
AUDIT = Path(os.getenv('HEART_AUDIT_OUT',str(Path('/project/yli4/ypan/audits/ugi_priority1_evidence') if UGI.is_dir() else Path.cwd()/'ugi_priority1_evidence'))).expanduser()
AUDIT.mkdir(parents=True,exist_ok=True)
PLOTS = AUDIT/'figures'; PLOTS.mkdir(parents=True,exist_ok=True)

RUN_FULL_AUDIT = True               # scans every records.csv.gz and builds resumable SQLite identity index
RUN_LMDB_PROBE = True               # real sampled payloads, only after the metadata audit
RUN_FORENSIC_EXAMPLES = True         # if duplicates or leakage, trace real molecular examples (second metadata pass)
RUN_PRODUCTION_DRY_RUN = False      # optional: explicitly run documented --dry-run; never submits
MAX_PROBE_RECORDS = 3500           # stratified, at most 3500 actual LMDB records
SQL_BATCH = 3000
SEED = 20261117

CORE = Chem.MolFromSmiles('C(=O)NCC(=O)N')  # diagnostic motif, NOT proof of Ugi synthetic origin

def utc_now(): return datetime.now(timezone.utc).isoformat()
def read_json(path):
    try: return json.loads(Path(path).read_text(encoding='utf-8'))
    except (OSError, ValueError): return None

def sha256(path):
    p=Path(path)
    if not p.is_file(): return None
    h=hashlib.sha256()
    with p.open('rb') as f:
        for ch in iter(lambda:f.read(1024*1024),b''):h.update(ch)
    return h.hexdigest()

def write_json(name,obj):
    path=AUDIT/name
    path.write_text(json.dumps(obj,ensure_ascii=False,indent=2,default=str)+'\n',encoding='utf-8')
    return path

def canonical_and_mol(s):
    if not isinstance(s,str) or not s.strip():return None,None
    try:
        m=Chem.MolFromSmiles(s.strip())
        if m is None: return None,None
        return Chem.MolToSmiles(m,canonical=True,isomericSmiles=True),m
    except Exception:return None,None

def fp(can): return hashlib.blake2b(can.encode('utf-8'),digest_size=16).digest()

def figsave(name):
    path=PLOTS/name
    plt.tight_layout();plt.savefig(path,dpi=175,bbox_inches='tight');plt.show();plt.close()
    print('Figure:',path)

print('UGI source exists:',UGI.is_dir(),UGI)
print('Stage-06 view exists:',VIEW.is_dir(),VIEW)
print('REPO exists:',REPO.is_dir(),REPO)
print('REPORTS (new, source untouched):',AUDIT)
print('RUN_FULL_AUDIT:',RUN_FULL_AUDIT,'| RUN_LMDB_PROBE:',RUN_LMDB_PROBE)

## 1 | Immutable source inventory and split integrity

**Evidence produced:** display declared record and shard counts, the train/validation allocation, and separate historical Stage-05 and Stage-06 statuses. Disjoint split membership at this stage means **disjoint shard assignment**, not necessarily disjoint molecular identity. Exact train/validation molecular leakage is evaluated in Section 3.


In [ ]:
manifest=read_json(UGI/'manifest.json')
splits=read_json(UGI/'splits.json')
qc=read_json(UGI/'quality_report.json')
view=read_json(VIEW/'final_dataset_view.json')
ready=read_json(UGI/'library_3d_ready.json')
SOURCE_FILES={'manifest':UGI/'manifest.json','splits':UGI/'splits.json','stage05_quality_report':UGI/'quality_report.json',
              'stage06_final_view':VIEW/'final_dataset_view.json','stage06_readiness_receipt':UGI/'library_3d_ready.json'}
SOURCE_HASHES={k:sha256(v) for k,v in SOURCE_FILES.items()}
INV={'generated_utc':utc_now(),'files':{k:{'path':str(v),'present':v.is_file(),'sha256':SOURCE_HASHES[k]} for k,v in SOURCE_FILES.items()},'issues':[]}
items=[]; SPLIT_BY_REL={}; MANIFEST_BY_REL={}; VIEW_BY_REL={}

if not manifest or not splits:
    INV['issues'].append('MISSING manifest.json and/or splits.json: run on HPC with mounted Ugi dataset')
else:
    entries=manifest.get('shards',[])
    MANIFEST_BY_REL={str(x['lmdb']):x for x in entries}
    t=set(map(str,splits.get('train',[])));v=set(map(str,splits.get('valid',[])))
    SPLIT_BY_REL={rel:split for split,refs in [('train',t),('valid',v)] for rel in refs}
    if len(MANIFEST_BY_REL)!=len(entries): INV['issues'].append('DUPLICATED shard names in manifest')
    if not t or not v:INV['issues'].append('Missing or empty train/valid set')
    if t & v:INV['issues'].append(f'Shard overlap between train and valid: {len(t&v)}')
    if t|v!=set(MANIFEST_BY_REL):INV['issues'].append('Split union does not equal manifest shard set')
    declared_total=sum(int(x['n_records']) for x in entries)
    if declared_total!=int(manifest.get('n_records',-1)):INV['issues'].append('Manifest n_records != per-shard sum')
    for split,refs in [('train',t),('valid',v)]:
        items.append({'split':split,'shards':len(refs),'declared_records':sum(int(MANIFEST_BY_REL[x]['n_records']) for x in refs if x in MANIFEST_BY_REL)})
    INV.update({'manifest_declared_records':manifest.get('n_records'),'manifest_shards':len(entries),'split_shards_disjoint':not bool(t&v),'split_complete':t|v==set(MANIFEST_BY_REL),'declared_per_split':items})
if view:
    VIEW_BY_REL={str(x['lmdb']):x for x in view.get('shards',[])}
    if manifest and set(VIEW_BY_REL)!=set(MANIFEST_BY_REL):INV['issues'].append('Stage-06 view shard set differs from manifest')
else:INV['issues'].append('Stage-06 final_dataset_view.json unavailable')
if not ready:INV['issues'].append('Stage-06 readiness receipt unavailable or unreadable')
INV['historical_stage05_passed']=qc.get('passed') if isinstance(qc,dict) else None
INV['view_index_validated_flag']=view.get('data_index_validated') if isinstance(view,dict) else None
INV['view_original_stage05_passed_flag']=view.get('source_original_quality_passed') if isinstance(view,dict) else None
INV['stage06_receipt_keys']=list(ready) if isinstance(ready,dict) else []
INV['stage06_view_keys']=list(view) if isinstance(view,dict) else []
if isinstance(ready,dict) and (ready.get('ready') is False or ready.get('passed') is False):
    INV['issues'].append('Stage-06 readiness explicitly reports failure')
if isinstance(view,dict) and view.get('data_index_validated') is False:
    INV['issues'].append('Stage-06 source data_index_validated explicitly false')
if isinstance(view,dict) and view.get('source_lmdb_dir'):
    declared_source=Path(view['source_lmdb_dir'])
    if declared_source.is_dir() and declared_source.resolve()!=UGI.resolve():
        INV['issues'].append(f'Stage-06 view references a DIFFERENT source root: {declared_source}')

write_json('01_source_inventory.json',INV)
print('Manifest records:', INV.get('manifest_declared_records'),'| shards:',INV.get('manifest_shards'))
print('Historical Stage-05 QC passed:',INV['historical_stage05_passed'])
print('Stage-06 final view index validated:',INV['view_index_validated_flag'])
print('Stage-06 receipt present:',bool(ready),'| keys:',INV['stage06_receipt_keys'])
if items:display(pd.DataFrame(items))
print('INVENTORY ISSUES:',INV['issues'] or 'none')
print('Evidence:',AUDIT/'01_source_inventory.json')

## 2 | Full Stage-06 3D sidecar census — every record, not just a readiness receipt

Read the actual **3D index bytes** referenced by `final_dataset_view.json`, without regenerating conformers or modifying LMDB files. Produce the full-library histogram of valid 3D conformers per record (1–10), zero-success counts by shard, and comparisons with record counts in the manifest. Section 3 subsequently verifies each index entry against its corresponding `n_3d_success` metadata value.


In [ ]:
sidecar_hist=Counter(); sidecar_rows=[]; sidecar_issues=[]; SIDE_COUNTS={}
if manifest and view:
    for rel,s in MANIFEST_BY_REL.items():
        if rel not in VIEW_BY_REL:
            sidecar_issues.append(f'MISSING view entry: {rel}'); continue
        idx_rel=VIEW_BY_REL[rel].get('index')
        if not idx_rel:
            sidecar_issues.append(f'MISSING index path: {rel}');continue
        idx_path=VIEW/idx_rel
        if not idx_path.is_file():
            sidecar_issues.append(f'MISSING index file: {idx_path}');continue
        counts=np.frombuffer(idx_path.read_bytes(),dtype=np.uint8)
        n_expected=int(s['n_records'])
        SIDE_COUNTS[rel]=counts
        if counts.size!=n_expected: sidecar_issues.append(f'{rel}: index has {counts.size}, expected {n_expected}')
        hist=np.bincount(counts,minlength=max(11,int(counts.max())+1 if counts.size else 11))
        for k,n in enumerate(hist):
            if n:sidecar_hist[k]+=int(n)
        if counts.size and (int(counts.min())<1 or int(counts.max())>10):
            sidecar_issues.append(f'{rel}: index out of 1-10 range ({int(counts.min())}-{int(counts.max())})')
        sidecar_rows.append({'shard':rel,'split':SPLIT_BY_REL.get(rel,'UNKNOWN'),'index_records':len(counts),
              'expected_records':n_expected,'n_3d_0':int(hist[0]),'n_3d_1':int(hist[1]),'n_3d_10':int(hist[10]),
              'mean_n3d':round(float(np.mean(counts)),3) if counts.size else None})
    del SIDE_COUNTS  # avoid accidentally holding all sidecar data in RAM; reread one shard in full audit
else:
    sidecar_issues.append('NOT_RUN: manifest and/or Stage-06 view missing')
SIDE_REPORT={'scope':'FULL_VIEW_SCAN' if manifest and view and len(sidecar_rows)==len(MANIFEST_BY_REL) else 'INCOMPLETE',
    'view_sha256':SOURCE_HASHES['stage06_final_view'],'declared_records':manifest.get('n_records') if manifest else None,
    'counted_records':sum(sidecar_hist.values()),'histogram_n3d':dict(sorted(sidecar_hist.items())),
    'shards_checked':len(sidecar_rows),'issues':sidecar_issues[:100], 'n_total_issues':len(sidecar_issues)}
if SIDE_REPORT['scope']=='FULL_VIEW_SCAN' and SIDE_REPORT['counted_records']!=SIDE_REPORT['declared_records']:
    sidecar_issues.append('Total indexed records != manifest declared records')
    SIDE_REPORT['n_total_issues']=len(sidecar_issues);SIDE_REPORT['issues']=sidecar_issues[:100]
write_json('02_3d_index_census.json',SIDE_REPORT)
if sidecar_rows:pd.DataFrame(sidecar_rows).to_csv(AUDIT/'02_3d_index_shards.csv',index=False)
print('INDEX CENSUS:',SIDE_REPORT['counted_records'],'records;',SIDE_REPORT['shards_checked'],'shards')
print('Zero-success slots:',sidecar_hist.get(0,0),'| out-of-range:',sum(v for k,v in sidecar_hist.items() if k>10))
print('Index problems:',SIDE_REPORT['n_total_issues'])
if sidecar_hist:
    plt.figure(figsize=(9,4));plt.bar(range(1,11),[sidecar_hist.get(i,0) for i in range(1,11)])
    plt.xlabel('Verified successful 3D conformer slots per molecule');plt.ylabel('Records (full index census)')
    plt.title('Stage-06: actual 3D-success distribution (all Ugi records)');plt.grid(axis='y',alpha=.2)
    figsave('01_full_3d_success_histogram.png')

## 3 | Full metadata ↔ index ↔ molecular identity audit (resume-safe SQLite)

- Read all `shard-*/records.csv.gz` files; verify row counts against the manifest and expected sequential record keys (`0, 1, …`).
- Verify that `smi` is parseable by RDKit, that `n_3d_success` is in the valid range (1–10), and that every value matches the corresponding **verified Stage-06 index byte**.
- Canonicalize with the same `canonical=True, isomericSmiles=True` policy and index molecular identity using a BLAKE2b-128 digest in SQLite. Hash collisions are extremely unlikely; for mathematically collision-free identity, store and compare complete canonical SMILES instead.
- Quantify **duplicate molecules across the entire library**, **molecules shared by train and validation**, suspicious disconnected fragments, Ugi-like core coverage, and the heavy-atom distribution.
- **Resume safely:** process each shard in an independent transaction and mark it complete only after successful processing. Reject reuse of a stale cache if the manifest, split, or Stage-06 view SHA-256 fingerprint changes.
- Full canonicalization is a substantial CPU workload. Display the full-library charts only after the corresponding scan has actually completed; never substitute estimated values.


In [ ]:
DB_PATH=AUDIT/'ugi_priority1_identity.sqlite3'

def db_connect():
    con=sqlite3.connect(str(DB_PATH),timeout=180)
    con.execute('PRAGMA journal_mode=WAL')
    con.execute('PRAGMA synchronous=NORMAL')
    con.execute('PRAGMA cache_size=-131072')
    con.execute('CREATE TABLE IF NOT EXISTS meta(k TEXT PRIMARY KEY,v TEXT NOT NULL)')
    con.execute('''CREATE TABLE IF NOT EXISTS molecules(
        hash BLOB PRIMARY KEY,n_train INTEGER NOT NULL,n_valid INTEGER NOT NULL,n_total INTEGER NOT NULL)''')
    con.execute('''CREATE TABLE IF NOT EXISTS done(
        shard TEXT PRIMARY KEY,n_records INTEGER NOT NULL,stats_json TEXT NOT NULL,samples_json TEXT NOT NULL)''')
    con.commit();return con

def gate_fingerprints(con):
    for k,v in [('manifest_sha256',SOURCE_HASHES['manifest']),('splits_sha256',SOURCE_HASHES['splits']),
                ('view_sha256',SOURCE_HASHES['stage06_final_view'])]:
        old=con.execute('SELECT v FROM meta WHERE k=?',(k,)).fetchone()
        if old and old[0]!=str(v):raise RuntimeError(f'Cache SOURCE CHANGED for {k}: use a NEW AUDIT directory; do not reuse this DB')
        con.execute('INSERT OR IGNORE INTO meta(k,v) VALUES (?,?)',(k,str(v)))
    con.commit()

def full_audit():
    if INV['issues']:raise RuntimeError('Cannot full audit until source inventory issues resolved: '+str(INV['issues'][:5]))
    if sidecar_issues:raise RuntimeError('Cannot full audit until sidecar issues resolved: '+str(sidecar_issues[:5]))
    con=db_connect()
    gate_fingerprints(con)
    done={rel:n for rel,n in con.execute('SELECT shard,n_records FROM done')}
    clock=time.monotonic()
    for index,(rel,entry) in enumerate(MANIFEST_BY_REL.items(),start=1):
        expected=int(entry['n_records'])
        meta=UGI/Path(rel).parent/'records.csv.gz'
        indexfile=VIEW/VIEW_BY_REL[rel]['index']
        if not meta.is_file():raise FileNotFoundError(meta)
        if not indexfile.is_file():raise FileNotFoundError(indexfile)
        meta_stat=meta.stat();index_stat=indexfile.stat()
        if rel in done:
            if done[rel]!=expected:raise RuntimeError('Cached n_records changed: '+rel)
            cached=json.loads(con.execute('SELECT stats_json FROM done WHERE shard=?',(rel,)).fetchone()[0])
            for field,actual in [('metadata_size',meta_stat.st_size),('metadata_mtime_ns',meta_stat.st_mtime_ns),
                                 ('index_size',index_stat.st_size),('index_mtime_ns',index_stat.st_mtime_ns)]:
                if field not in cached or cached[field]!=actual:
                    raise RuntimeError(f'Cached shard source changed ({rel} / {field}). Use a NEW AUDIT directory.')
            continue
        counts=np.frombuffer((VIEW/VIEW_BY_REL[rel]['index']).read_bytes(),dtype=np.uint8)
        if len(counts)!=expected:raise RuntimeError(f'Index count mismatch {rel}')
        split=SPLIT_BY_REL[rel]; train_bit=int(split=='train'); valid_bit=int(split=='valid')
        rng=random.Random(SEED+int.from_bytes(hashlib.blake2b(rel.encode(),digest_size=4).digest(),'big'))
        chosen={0,expected-1}
        if expected>2:chosen.add(rng.randrange(expected))
        stats={'shard':rel,'split':split,'expected':expected,'read':0,'noncontiguous_keys':0,
               'bad_3d_count':0,'sidecar_disagreement':0,'invalid_smiles':0,'fragmented_smiles':0,
               'without_ugi_like_motif':0,'heavy_over_256':0,'n3d_hist':{},'heavy_hist':{},
               'max_heavy_atoms':0,'examples':{},
               'metadata_size':meta_stat.st_size,'metadata_mtime_ns':meta_stat.st_mtime_ns,
               'index_size':index_stat.st_size,'index_mtime_ns':index_stat.st_mtime_ns}
        hist3d=Counter();heavyhist=Counter();samples=[];low3d_collected=0;batch=[]
        with con:  # fail-safe transaction for this exact shard
            with gzip.open(meta,'rt',encoding='utf-8',newline='') as f:
                reader=csv.DictReader(f)
                if not {'key','smi','n_3d_success'}.issubset(set(reader.fieldnames or [])):
                    raise ValueError('Missing metadata fields in '+str(meta))
                for i,row in enumerate(reader):
                    stats['read']+=1
                    try:key=int(row['key'])
                    except Exception:key=-1
                    if key!=i:
                        stats['noncontiguous_keys']+=1
                        stats['examples'].setdefault('noncontiguous_key',{'row':i,'key':row.get('key')})
                    try:n3d=int(row['n_3d_success'])
                    except Exception:n3d=-1
                    hist3d[n3d]+=1
                    if not (1<=n3d<=10):
                        stats['bad_3d_count']+=1
                        stats['examples'].setdefault('bad_3d_count',{'row':i,'n3d':n3d})
                    if i>=len(counts) or n3d!=int(counts[i]):
                        stats['sidecar_disagreement']+=1
                        stats['examples'].setdefault('sidecar_disagreement',{'row':i,'metadata_n3d':n3d,'view_index_n3d':int(counts[i]) if i<len(counts) else None})
                    can,mol=canonical_and_mol(row.get('smi'))
                    if can is None:
                        stats['invalid_smiles']+=1
                        stats['examples'].setdefault('invalid_smiles',{'row':i,'smi':str(row.get('smi'))[:150]})
                    else:
                        if '.' in can:
                            stats['fragmented_smiles']+=1
                            stats['examples'].setdefault('fragmented_smiles',{'row':i,'smi':can[:150]})
                        if not mol.HasSubstructMatch(CORE):
                            stats['without_ugi_like_motif']+=1
                            stats['examples'].setdefault('without_ugi_like_motif',{'row':i,'smi':can[:150]})
                        heavy=mol.GetNumHeavyAtoms();heavyhist[min(heavy,513)]+=1
                        stats['max_heavy_atoms']=max(stats['max_heavy_atoms'],heavy)
                        if heavy>256:
                            stats['heavy_over_256']+=1
                            stats['examples'].setdefault('heavy_over_256',{'row':i,'heavy':heavy,'smi':can[:100]})
                        batch.append((fp(can),train_bit,valid_bit))
                        if len(batch)>=SQL_BATCH:
                            con.executemany('''INSERT INTO molecules(hash,n_train,n_valid,n_total) VALUES(?,?,?,1)
                                ON CONFLICT(hash) DO UPDATE SET n_train=n_train+excluded.n_train,
                                n_valid=n_valid+excluded.n_valid,n_total=n_total+1''',batch)
                            batch.clear()
                    if i in chosen or (n3d<=2 and low3d_collected<2 and n3d>=1):
                        samples.append({'shard':rel,'split':split,'key':i,'n3d':n3d,'smi':row.get('smi',''),
                                        'selection':'random_or_edge' if i in chosen else 'low_n3d'})
                        if n3d<=2 and i not in chosen:low3d_collected+=1
            if batch:
                con.executemany('''INSERT INTO molecules(hash,n_train,n_valid,n_total) VALUES(?,?,?,1)
                    ON CONFLICT(hash) DO UPDATE SET n_train=n_train+excluded.n_train,
                    n_valid=n_valid+excluded.n_valid,n_total=n_total+1''',batch)
            stats['n3d_hist']={str(k):int(v) for k,v in hist3d.items()}
            stats['heavy_hist']={str(k):int(v) for k,v in heavyhist.items()}
            con.execute('INSERT INTO done(shard,n_records,stats_json,samples_json) VALUES(?,?,?,?)',
                        (rel,stats['read'],json.dumps(stats),json.dumps(samples)))
        if index%25==0 or index==len(MANIFEST_BY_REL):
            print(f'Completed {index}/{len(MANIFEST_BY_REL)} shards; this session {time.monotonic()-clock:.0f}s',flush=True)
    con.close()

def summarize_full_audit():
    if not DB_PATH.is_file():return None
    con=db_connect()
    expected=len(MANIFEST_BY_REL)
    done_rows=con.execute('SELECT shard,n_records,stats_json FROM done').fetchall()
    allstats=[json.loads(j) for _,_,j in done_rows]
    count_rows=sum(st['read'] for st in allstats)
    total=Counter();n3d=Counter();heavy=Counter();bad_examples=[]
    for s in allstats:
        for fld in ['noncontiguous_keys','bad_3d_count','sidecar_disagreement','invalid_smiles',
                    'fragmented_smiles','without_ugi_like_motif','heavy_over_256']:
            total[fld]+=int(s.get(fld,0))
        for k,v in s.get('n3d_hist',{}).items():n3d[k]+=v
        for k,v in s.get('heavy_hist',{}).items():heavy[k]+=v
        for t,example in s.get('examples',{}).items():
            if len(bad_examples)<150:bad_examples.append({'shard':s['shard'],'issue':t,**example})
    sql_count=con.execute('''SELECT COUNT(*),COALESCE(SUM(n_total-1),0),
          COALESCE(SUM(CASE WHEN n_train>0 AND n_valid>0 THEN 1 ELSE 0 END),0),
          COALESCE(SUM(CASE WHEN n_train>1 AND n_valid=0 THEN 1 ELSE 0 END),0),
          COALESCE(SUM(CASE WHEN n_valid>1 AND n_train=0 THEN 1 ELSE 0 END),0)
          FROM molecules''').fetchone()
    uniq,extra,cross,tr_dup,va_dup=map(int,sql_count)
    full=bool(manifest and len(done_rows)==expected and count_rows==int(manifest['n_records']) and
              all(st['read']==st['expected'] for st in allstats))
    report={'generated_utc':utc_now(),'scope':'FULL' if full else 'INCOMPLETE',
       'expected_shards':expected,'audited_shards':len(done_rows),
       'expected_records':int(manifest['n_records']) if manifest else None,'audited_records':count_rows,
       'unique_canonical_molecules_hash_based':uniq,'duplicate_excess_records':extra,
       'canonical_train_valid_intersection':cross,'train_only_duplicate_molecules':tr_dup,
       'valid_only_duplicate_molecules':va_dup,'issues_full_counts':dict(total),
       'n3d_histogram':dict(n3d),'heavy_atom_histogram_cap513':dict(heavy),
       'source_hashes':SOURCE_HASHES,'identity_accounting_matches':bool(uniq+extra==count_rows-total.get('invalid_smiles',0)),
       'note':'BLAKE2b-128 canonical isomeric RDKit identity. FULL means all metadata visited, not automatic PASS.'}
    write_json('03_full_molecular_audit.json',report)
    pd.DataFrame(allstats).drop(columns=['n3d_hist','heavy_hist','examples'],errors='ignore').to_csv(AUDIT/'03_per_shard_stats.csv',index=False)
    pd.DataFrame(bad_examples).to_csv(AUDIT/'03_anomaly_examples.csv',index=False)
    con.close();return report

if RUN_FULL_AUDIT and manifest and splits and view:
    full_audit()
if manifest and splits and view:UGI_REPORT=summarize_full_audit()
else:UGI_REPORT=None
if UGI_REPORT:
    display(pd.DataFrame([{'Metric':k,'Measured':UGI_REPORT[k]} for k in [
       'scope','expected_shards','audited_shards','expected_records','audited_records',
       'unique_canonical_molecules_hash_based','duplicate_excess_records','canonical_train_valid_intersection']]))
    display(pd.DataFrame([{'Issue':k,'Actual records':v} for k,v in UGI_REPORT['issues_full_counts'].items()]))
else: print('Full metadata audit not available. NO CLAIM of training readiness.')

## 3B | Forensic examples of actual molecules when duplication or leakage is detected

If the full scan finds canonical duplicates or train/validation leakage, select a small number of matching molecular digests from SQLite and revisit `records.csv.gz` to identify the actual SMILES, source shard, and record key. This is **traceable molecular evidence**, not merely a duplicate count, and it does not alter training data. If no duplicates or leakage are found, explicitly report that there is no forensic target to display.


In [ ]:
FORENSIC_ROWS=[]
if UGI_REPORT and UGI_REPORT['scope']=='FULL' and RUN_FORENSIC_EXAMPLES:
    con=db_connect()
    targets={}
    cross=con.execute('SELECT hash,n_train,n_valid,n_total FROM molecules WHERE n_train>0 AND n_valid>0 ORDER BY hash LIMIT 8').fetchall()
    duplicates=con.execute('SELECT hash,n_train,n_valid,n_total FROM molecules WHERE n_total>1 ORDER BY hash LIMIT 8').fetchall()
    for label,rows in [('train_valid_leakage',cross),('duplicate_molecule',duplicates)]:
        for h,nt,nv,tot in rows:
            t=targets.setdefault(h,{'kinds':set(),'n_train':nt,'n_valid':nv,'n_total':tot,'hits':[]})
            t['kinds'].add(label)
    con.close()
    if targets:
        for rel,entry in MANIFEST_BY_REL.items():
            if all((len(t['hits'])>=2 and
                    ('train_valid_leakage' not in t['kinds'] or
                     {'train','valid'}.issubset({x['split'] for x in t['hits']}))) for t in targets.values()):
                break
            path=UGI/Path(rel).parent/'records.csv.gz'
            with gzip.open(path,'rt',encoding='utf-8',newline='') as f:
                for row in csv.DictReader(f):
                    can,_=canonical_and_mol(row.get('smi'))
                    if not can:continue
                    h=fp(can)
                    if h in targets:
                        t=targets[h]; split=SPLIT_BY_REL[rel]
                        need=('train_valid_leakage' in t['kinds'] and
                              split not in {x['split'] for x in t['hits']})
                        if len(t['hits'])<2 or need:
                            t['hits'].append({'split':split,'shard':rel,'key':row.get('key'),'canonical_smiles':can})
        for h,t in targets.items():
            for x in t['hits']:
                FORENSIC_ROWS.append({'categories':','.join(sorted(t['kinds'])),'digest_hex':h.hex(),
                    'canonical_smiles':x['canonical_smiles'],'n_train':t['n_train'],'n_valid':t['n_valid'],
                    'n_total':t['n_total'],'split':x['split'],'shard':x['shard'],'key':x['key']})
        print('Selected molecular identities:',len(targets),'| source occurrences recovered:',len(FORENSIC_ROWS))
    else:print('No canonical duplicate or cross-split leakage identities in the FULL index.')
else:
    print('Forensic example recovery unavailable or disabled; do not interpret as 0 duplicates.')
pd.DataFrame(FORENSIC_ROWS,columns=['categories','digest_hex','canonical_smiles','n_train','n_valid','n_total','split','shard','key']).to_csv(AUDIT/'03b_molecular_forensic_examples.csv',index=False)
if FORENSIC_ROWS:display(pd.DataFrame(FORENSIC_ROWS).head(24))

## 4 | Visual evidence from the completed full metadata audit

**Only plot measurements obtained from the full dataset.** Figures cover train versus validation records, number of usable 3D conformers, molecular size, Ugi-like motif coverage, duplicate excess, and cross-split molecular leakage. Values come from the actual metadata and SQLite audit, **not from extrapolating a sample**.


In [ ]:
if UGI_REPORT and UGI_REPORT['scope']=='FULL':
    n3d={int(k):int(v) for k,v in UGI_REPORT['n3d_histogram'].items()}
    heavy={int(k):int(v) for k,v in UGI_REPORT['heavy_atom_histogram_cap513'].items()}
    plt.figure(figsize=(9,4))
    plt.bar(range(1,11),[n3d.get(k,0) for k in range(1,11)])
    plt.xlabel('Successful 3D slots (metadata verified against Stage-06 index)')
    plt.ylabel('Full-library record count');plt.grid(axis='y',alpha=.2)
    plt.title('Actual usable 3D conformers per Ugi record');figsave('02_metadata_3d_success.png')
    plt.figure(figsize=(10,4))
    x=np.arange(1,202)
    plt.bar(x,[heavy.get(int(k),0) for k in x],width=1)
    plt.xlabel('Number of heavy atoms (1–201; >201 reported separately)')
    plt.ylabel('Full-library record count');plt.title('Ugi molecular size distribution');figsave('03_heavy_atom_distribution.png')
    vals=[UGI_REPORT['unique_canonical_molecules_hash_based'],UGI_REPORT['duplicate_excess_records'],
          UGI_REPORT['canonical_train_valid_intersection']]
    plt.figure(figsize=(8,4));plt.bar(['Unique molecules','Duplicate excess','Train–valid shared'],vals)
    plt.ylabel('Count');plt.title('Canonical molecular identity: evidence, not just a PASS label')
    plt.grid(axis='y',alpha=.2);figsave('04_identity_overlap_counts.png')
    fractions=UGI_REPORT['issues_full_counts']
    labels=['RDKit invalid','Index mismatch','No successful 3D','Multi-fragment','No Ugi-like core']
    nums=[fractions.get('invalid_smiles',0),fractions.get('sidecar_disagreement',0),
          fractions.get('bad_3d_count',0),fractions.get('fragmented_smiles',0),
          fractions.get('without_ugi_like_motif',0)]
    plt.figure(figsize=(9,4));plt.barh(labels[::-1],nums[::-1])
    plt.xlabel('Affected records (full metadata audit)');plt.title('Chemical and 3D quality anomalies')
    plt.grid(axis='x',alpha=.2);figsave('05_full_audit_anomalies.png')

    # Actual train/valid 3D distribution, aggregated from full per-shard metadata.
    con=db_connect(); hist_by_split={'train':Counter(),'valid':Counter()}; heavy_by_split={'train':Counter(),'valid':Counter()}
    for (raw,) in con.execute('SELECT stats_json FROM done'):
        sh=json.loads(raw);sp=sh['split']
        if sp not in hist_by_split:continue
        hist_by_split[sp].update({int(k):int(v) for k,v in sh.get('n3d_hist',{}).items()})
        heavy_by_split[sp].update({int(k):int(v) for k,v in sh.get('heavy_hist',{}).items()})
    con.close()
    x=np.arange(1,11); width=.38
    plt.figure(figsize=(9,4))
    for sp,offset in [('train',-width/2),('valid',width/2)]:
        h=hist_by_split[sp];den=max(1,sum(h.values()))
        plt.bar(x+offset,[100*h.get(int(k),0)/den for k in x],width,label=sp)
    plt.xlabel('Number of verified successful 3D slots');plt.ylabel('% of records in each split')
    plt.title('Train vs valid: 3D-quality distributions, full records');plt.legend()
    plt.grid(axis='y',alpha=.2);figsave('05b_train_valid_3d_distribution.png')
    x=np.arange(1,202)
    plt.figure(figsize=(9,4))
    for sp in ['train','valid']:
        h=heavy_by_split[sp];den=max(1,sum(h.values()))
        plt.plot(x,[100*h.get(int(k),0)/den for k in x],label=sp)
    plt.xlabel('Heavy-atom count (1–201)');plt.ylabel('% of records within each split')
    plt.title('Train vs valid: molecular size distribution, full records');plt.legend()
    plt.grid(alpha=.2);figsave('05c_train_valid_heavy_atoms.png')

else: print('Figures withheld until FULL metadata audit is complete; partial != clean.')

## 5 | Real LMDB payload validation and genuine 3D molecular visualization

**Sections 2 and 3 establish the integrity of the sidecar and metadata. This section opens the original LMDB files and tests actual molecular payloads:**

- Verify the numeric key resolves to a record; its pickle payload contains `smi`, `atoms`, and `coordinates`; and its SMILES agrees with the corresponding metadata identity.
- Check that each of the ten coordinate slots has shape `(n_atoms, 3)`, all coordinate values are finite, and atom symbols agree with the RDKit molecule after adding hydrogens.
- **Treat only the first `n_3d_success` slots as validated 3D conformers.** Later slots may be fallbacks and must not be selected by the production data loader.
- Visualize a **real 3D molecular record loaded from the HPC LMDB**, not a separately regenerated example. A few 3D figures cannot prove the validity of all 15M records; interpret them alongside the complete 3D index and metadata census from Sections 2 and 3.

Set `MAX_PROBE_RECORDS` to control the sample size. The default selection draws from multiple shards, both splits, and records with few successful 3D conformers. All observed errors are retained in a detailed CSV report.


In [ ]:
def probe_real_lmdb():
    try:import lmdb
    except ImportError as e: raise RuntimeError('Please install lmdb in the HPC notebook kernel') from e
    if not UGI_REPORT or UGI_REPORT['scope']!='FULL':
        return {'status':'NOT_RUN','reason':'Full metadata audit required first'}
    con=db_connect()
    choices=[]
    for (raw,) in con.execute('SELECT samples_json FROM done'):
        choices.extend(json.loads(raw))
    con.close()
    rng=random.Random(SEED)
    if len(choices)>MAX_PROBE_RECORDS:
        # Keep deliberate low-success examples and stratify remaining by source split.
        risk=[v for v in choices if v['selection']=='low_n3d']
        normal=[v for v in choices if v['selection']!='low_n3d']
        risks=rng.sample(risk,min(len(risk),MAX_PROBE_RECORDS//3))
        remaining=MAX_PROBE_RECORDS-len(risks)
        normal=rng.sample(normal,min(len(normal),remaining))
        choices=risks+normal
    if {'train','valid'}.issubset(set(SPLIT_BY_REL.values())) and {v['split'] for v in choices}!={'train','valid'}:
        raise RuntimeError('LMDB probe misses train or validation; increase MAX_PROBE_RECORDS or inspect split sampling')
    by_shard=defaultdict(list)
    for item in choices:by_shard[item['shard']].append(item)
    issues=[];record_rows=[]; representative=None
    for rel,subset in by_shard.items():
        try:
            env=lmdb.open(str(UGI/rel),subdir=False,readonly=True,lock=False,readahead=False,max_readers=4)
        except Exception as exc:
            for x in subset:issues.append({'shard':rel,'key':x['key'],'issue':'LMDB_open_failed','detail':repr(exc)})
            continue
        try:
            with env.begin(write=False) as txn:
                for spec in subset:
                    errs=[];key=spec['key'];item=None;zrange=None
                    try:
                        raw=txn.get(str(key).encode('ascii'))
                        if raw is None:raise KeyError('Missing numeric LMDB key')
                        item=pickle.loads(raw)
                        if not isinstance(item,dict):raise TypeError('LMDB record is not dict')
                        smi=item.get('smi');atoms=item.get('atoms');coords=item.get('coordinates')
                        can,mol=canonical_and_mol(smi);can_ref,_=canonical_and_mol(spec['smi'])
                        if can is None or can!=can_ref:errs.append('SMILES_identity_mismatch')
                        if not isinstance(atoms,(list,tuple)):errs.append('atoms_missing_or_wrong_type')
                        if not isinstance(coords,(list,tuple)) or len(coords)!=10:errs.append('coordinate_slots_not_10')
                        if mol is not None and isinstance(atoms,(list,tuple)):
                            reference_atoms=[a.GetSymbol() for a in Chem.AddHs(mol).GetAtoms()]
                            if list(atoms)!=reference_atoms:errs.append('atoms_differ_from_RDKit_AddHs')
                        n3d=int(spec['n3d']);conformer=None;zrange=None
                        if isinstance(coords,(list,tuple)):
                            for ix,c in enumerate(coords):
                                try:
                                    arr=np.asarray(c,dtype=float)
                                    if not isinstance(atoms,(list,tuple)) or arr.shape!=(len(atoms),3):errs.append(f'coordinate_shape_bad_slot_{ix}')
                                    if not np.all(np.isfinite(arr)):errs.append(f'nonfinite_slot_{ix}')
                                    if arr.size and np.max(np.abs(arr))>1e4:errs.append(f'extreme_coordinate_slot_{ix}')
                                    if ix==0:
                                        conformer=arr
                                        if arr.shape[1:]==(3,):zrange=float(np.ptp(arr[:,2]))
                                except Exception as e:errs.append(f'coordinates_parse_failed_slot_{ix}')
                        if not (1<=n3d<=10):errs.append('invalid_n3d_selected')
                        if not errs and representative is None and conformer is not None and zrange is not None and zrange>.01:
                            representative=(spec,atoms,conformer,mol)
                    except Exception as exc:
                        errs.append('unreadable_payload:'+repr(exc)[:160])
                    record_rows.append({'shard':rel,'split':spec['split'],'key':key,'n3d':spec['n3d'],
                                        'selection':spec['selection'],'errors':';'.join(errs),
                                        'z_range_first_success':zrange if item is not None else None})
                    for e in errs:issues.append({'shard':rel,'key':key,'issue':e,'smi':str(spec.get('smi'))[:180]})
        finally:env.close()
    pd.DataFrame(record_rows).to_csv(AUDIT/'05_real_lmdb_probe_rows.csv',index=False)
    pd.DataFrame(issues,columns=['shard','key','issue','smi']).to_csv(AUDIT/'05_real_lmdb_probe_errors.csv',index=False)
    report={'status':'COMPLETE' if len(record_rows)==len(choices) else 'PARTIAL','generated_utc':utc_now(),'requested_probes':len(choices),
            'actual_probes':len(record_rows),'distinct_shards':len(by_shard),'errors':len(issues),
            'error_counts':dict(Counter(i['issue'] for i in issues)),
            'note':'Sampled LMDB payload probe; does not replace full Stage-06 view verification.'}
    write_json('05_real_lmdb_probe_summary.json',report)
    if representative:
        spec,atoms,coords,mol=representative
        fig=plt.figure(figsize=(7,6));ax=fig.add_subplot(111,projection='3d')
        atom_colors={'C':'gray','H':'lightgray','N':'blue','O':'red','S':'orange','P':'purple'}
        colors=[atom_colors.get(a,'green') for a in atoms]
        ax.scatter(coords[:,0],coords[:,1],coords[:,2],c=colors,s=[40 if a!='H' else 8 for a in atoms],alpha=.9)
        if mol is not None:
            molH=Chem.AddHs(mol)
            if molH.GetNumAtoms()==len(coords):
                for b in molH.GetBonds():
                    i,j=b.GetBeginAtomIdx(),b.GetEndAtomIdx()
                    ax.plot([coords[i,0],coords[j,0]],[coords[i,1],coords[j,1]],
                            [coords[i,2],coords[j,2]],color='silver',linewidth=.6,alpha=.6)
        ax.set_title(f'Actual Stage-06 valid 3D prefix: {spec["shard"]} key={spec["key"]}')
        ax.set_xlabel('X (Å)');ax.set_ylabel('Y (Å)');ax.set_zlabel('Z (Å)')
        figsave('06_real_LMDB_molecular_conformer.png')
        report['representative_real_structure']={'shard':spec['shard'],'key':spec['key'],'n3d':spec['n3d']}
        write_json('05_real_lmdb_probe_summary.json',report)
    print('ACTUAL LMDB PROBE:',report)
    return report

if RUN_LMDB_PROBE and UGI_REPORT and UGI_REPORT['scope']=='FULL':
    LMDB_REPORT=probe_real_lmdb()
else:
    LMDB_REPORT=read_json(AUDIT/'05_real_lmdb_probe_summary.json')
    if not LMDB_REPORT:LMDB_REPORT={'status':'NOT_RUN','reason':'RUN_LMDB_PROBE disabled or full audit incomplete'}
print('Payload probe:',LMDB_REPORT.get('status'),'| errors:',LMDB_REPORT.get('errors','UNKNOWN'))

## 6 | Production loader and training readiness — separate from molecular data validity

Even if the data and 3D index checks pass, we must confirm that **the production loader actually samples only the 3D slots permitted by `n_3d_success`**. If the original LMDB still includes fallback slots, using a loader without Stage-06 protections could reintroduce invalid coordinates.

This section inventories relevant source-code references and the General checkpoint, with an optional `bin/heart continual --dry-run` check. **Static code searches provide audit clues, not proof of correct runtime behavior.** `RUN_PRODUCTION_DRY_RUN=False` by default, so this notebook does not submit a training job.


In [ ]:
LOADER={'general_checkpoint_exists':GENERAL_CKPT.is_file(),'general_checkpoint_path':str(GENERAL_CKPT),
        'repo_exists':REPO.is_dir(),'code_evidence':[],'dry_run':'NOT_RUN'}
if REPO.is_dir():
    tokens=['n_3d_success','final_dataset_view','library_3d_ready','3d_success','valid_3d','index_file']
    for p in REPO.rglob('*.py'):
        if len(LOADER['code_evidence'])>=60:break
        try:
            if p.stat().st_size>800000 or any(part in {'site-packages','.venv','third_party','.git'} for part in p.parts):continue
            for lineno,line in enumerate(p.read_text(encoding='utf-8',errors='ignore').splitlines(),1):
                if any(x in line for x in tokens):
                    LOADER['code_evidence'].append({'path':str(p),'line_number':lineno,'text':line.strip()[:190]})
                    if len(LOADER['code_evidence'])>=60:break
        except (OSError,PermissionError):pass
if RUN_PRODUCTION_DRY_RUN:
    launcher=REPO/'bin/heart'
    if not launcher.is_file():LOADER['dry_run']='FAILED: missing bin/heart'
    else:
        try:
            r=subprocess.run([str(launcher),'continual','--dry-run'],cwd=str(REPO),capture_output=True,text=True,timeout=900)
            (AUDIT/'06_production_dry_run.log').write_text(r.stdout+'\nSTDERR\n'+r.stderr)
            LOADER['dry_run']='EXIT_0' if r.returncode==0 else 'EXIT_'+str(r.returncode)
        except Exception as e:LOADER['dry_run']='ERROR: '+repr(e)
write_json('06_production_loader_review.json',LOADER)
print('General checkpoint exists:',LOADER['general_checkpoint_exists'])
print('Static sidecar/valid-3D code hits:',len(LOADER['code_evidence']))
print('dry-run:',LOADER['dry_run'])
if LOADER['code_evidence']:display(pd.DataFrame(LOADER['code_evidence'][:20]))
print('IMPORTANT: A code hit or exit 0 alone cannot prove that the training iterator uses only successful 3D slots. Inspect actual run configuration / sampled batch.')

## 7 | Evidence-based readiness decision: counts, charts, anomalies, and next actions

Report only conclusions supported by checks that actually ran:

- **`HOLD`:** confirmed defects in source integrity, chemical validity, sidecars, sampled LMDB payloads, or train/validation molecular leakage.
- **`INCOMPLETE`:** missing datasets, unreadable evidence, or unfinished scans; do **not** assume the dataset passed.
- **`REVIEW`:** key data checks are satisfactory, but duplicate sampling policy, historical Stage-05 failures, or production-loader behavior still require review.
- **`GO_DATA`:** the complete molecular / 3D metadata audit and sampled payload check have identified no blocking data issue. This means **data-ready for a carefully controlled dry-run**, not that the model already learns lipid delivery.

Production execution readiness is reported separately as `PENDING_RUNTIME_VERIFICATION`, so a dataset-quality result cannot be mistaken for authorization to launch a training run.


In [ ]:
blockers=[];unknown=[];reviews=[];context_notes=[];production_reviews=[];positives=[]
for issue in INV['issues']:
    if issue.startswith('MISSING ') or 'unavailable' in issue:unknown.append(issue)
    else:blockers.append(issue)
for issue in sidecar_issues[:20]:
    if issue.startswith('NOT_RUN') or issue.startswith('MISSING '):unknown.append(issue)
    else:blockers.append(issue)
if not UGI_REPORT or UGI_REPORT['scope']!='FULL':
    unknown.append('Full 15M metadata canonical scan missing or incomplete; no exact duplicate / leakage conclusion')
else:
    u=UGI_REPORT
    if not u.get('identity_accounting_matches',False):
        blockers.append('Canonical identity count mismatch: unique+duplicate excess != valid SMILES records')
    if u['audited_records']==u['expected_records'] and u['audited_shards']==u['expected_shards']:
        positives.append(f'All {u["audited_records"]:,} metadata records in {u["audited_shards"]:,} shards audited')
    for name in ['noncontiguous_keys','bad_3d_count','sidecar_disagreement','invalid_smiles','fragmented_smiles']:
        count=u['issues_full_counts'].get(name,0)
        if count:blockers.append(f'{name}: {count:,} affected records')
        else:positives.append(f'Zero {name} across full metadata scan')
    if u['canonical_train_valid_intersection']>0:
        blockers.append(f'Canonical train/validation leakage: {u["canonical_train_valid_intersection"]:,} molecules')
    else:positives.append('Zero canonical molecular overlap between train and validation')
    if u['duplicate_excess_records']>0:
        reviews.append(f'Train/library duplicate excess = {u["duplicate_excess_records"]:,}; document sampling/dedup policy before production')
    if u['issues_full_counts'].get('without_ugi_like_motif',0)>0:
        reviews.append(f'{u["issues_full_counts"]["without_ugi_like_motif"]:,} records do not contain broad Ugi-like motif; review chemistry, this motif is only diagnostic')
    if u['issues_full_counts'].get('heavy_over_256',0)>0:
        reviews.append(f'{u["issues_full_counts"]["heavy_over_256"]:,} molecules exceed 256 heavy atoms; verify actual production model max atoms (256 is not automatically HEART limit)')
if not SIDE_REPORT or SIDE_REPORT['scope']!='FULL_VIEW_SCAN':unknown.append('Full Stage-06 view census not completed')
elif SIDE_REPORT.get('n_total_issues',0)==0 and SIDE_REPORT['counted_records']==SIDE_REPORT['declared_records']:
    positives.append('Stage-06 successful-3D sidecar index exists, has valid range, and covers all declared records')
if LMDB_REPORT.get('status')!='COMPLETE':unknown.append('Real LMDB payload probe not run')
elif LMDB_REPORT.get('errors',0)>0:blockers.append(f'{LMDB_REPORT["errors"]} actual LMDB probe errors')
else:positives.append(f'{LMDB_REPORT["actual_probes"]:,} actual LMDB sampled payloads passed structural/coordinate checks')
if qc and qc.get('passed') is False:
    reviews.append('Historical Stage-05 QC marked FAIL; confirm its precise cause and documented Stage-06 resolution (do not overwrite history)')
if not LOADER['general_checkpoint_exists']:
    production_reviews.append('Expected General checkpoint not located at configured path')
context_notes.append('Whole-shard train/valid split is not reagent-holdout or scaffold-holdout; interpret ID validation accordingly')
if LOADER['dry_run']!='EXIT_0':production_reviews.append('Production loader/dry-run has not been dynamically verified on this dataset')
else:positives.append('Documented bin/heart continual --dry-run exited successfully; inspect its logged input configuration')

if blockers:gate='HOLD'
elif unknown:gate='INCOMPLETE'
elif reviews:gate='REVIEW'
else:gate='GO_DATA'
PRODUCTION_GATE='PENDING_RUNTIME_VERIFICATION' if LOADER['dry_run']!='EXIT_0' else 'INSPECT_DRY_RUN_EVIDENCE'
REPORT={'generated_utc':utc_now(),'data_gate':gate,'production_gate':PRODUCTION_GATE,
        'blockers':blockers,'unknown':unknown,'reviews':reviews,'production_reviews':production_reviews,'context_notes':context_notes,'positives':positives,
        'inventory':INV,'full_ugi':UGI_REPORT,'sidecar':SIDE_REPORT,'payload_probe':LMDB_REPORT,
        'production_check':LOADER,'figures':sorted(p.name for p in PLOTS.glob('*.png')),
        'disclaimer':'Data readiness cannot prove continual-pretraining performance or LNP delivery efficacy. No training submitted.'}
write_json('07_priority1_evidence_decision.json',REPORT)
lines=['# HEART Ugi-4CR | Priority 1 evidence decision',
       f'**Data gate: {gate}** | **Production gate: {PRODUCTION_GATE}**',
       f'Generated UTC: {REPORT["generated_utc"]}','',
       '## Actual measurement summary',
       f'- Expected / audited: {UGI_REPORT.get("audited_records") if UGI_REPORT else "UNKNOWN"} / {UGI_REPORT.get("expected_records") if UGI_REPORT else "UNKNOWN"} records',
       f'- Unique canonical molecules: {UGI_REPORT.get("unique_canonical_molecules_hash_based") if UGI_REPORT else "UNKNOWN"}',
       f'- Duplicate excess: {UGI_REPORT.get("duplicate_excess_records") if UGI_REPORT else "UNKNOWN"}',
       f'- Train/valid shared molecules: {UGI_REPORT.get("canonical_train_valid_intersection") if UGI_REPORT else "UNKNOWN"}',
       f'- 3D sidecar indexed records: {SIDE_REPORT.get("counted_records")}',
       f'- Real LMDB payload probes: {LMDB_REPORT.get("actual_probes","NOT_RUN")} | errors: {LMDB_REPORT.get("errors","UNKNOWN")}',
       '','## Positive evidence']
lines.extend('- '+x for x in positives or ['None established yet'])
lines.extend(['','## Confirmed blockers']);lines.extend('- '+x for x in blockers or ['None'])
lines.extend(['','## Still unknown']);lines.extend('- '+x for x in unknown or ['None'])
lines.extend(['','## Review before production']);lines.extend('- '+x for x in reviews or ['None'])
lines.extend(['','## Production-only review']);lines.extend('- '+x for x in production_reviews or ['None'])
lines.extend(['','## Context / limitations']);lines.extend('- '+x for x in context_notes or ['None'])
lines.extend(['','## Saved figures']);lines.extend('- '+x for x in REPORT['figures'] or ['No figures yet'])
lines.extend(['','## Interpretation',
  'Stage-06 safe 3D-prefix policy must be enforced by the actual training loader; a valid sidecar is necessary but not sufficient.',
  'Stage-05 historical status is not rewritten. No automatic dataset modification or training submission.'])
(AUDIT/'07_priority1_evidence_decision.md').write_text('\n'.join(lines)+'\n',encoding='utf-8')
display(Markdown('\n'.join(lines)))
print('Final evidence:',AUDIT/'07_priority1_evidence_decision.json')

## 8 | How to present these findings to the PI

**Claims supported by completed tests (replace placeholders with actual HPC results):**

- “We scanned **X records across Y shards**. The observed Stage-06 3D-success distribution is shown in the figures, with **Z metadata-versus-sidecar mismatches**.”
- “The library contains **A unique canonical molecules**, **B excess duplicate records**, and **C molecules shared between training and validation**.”
- “We inspected **N real LMDB records**, observed **E payload errors**, and retained molecular coordinate visualizations and forensic anomaly examples.”
- “The existence of the General checkpoint and production-loader runtime readiness were assessed separately.”

**Claims that the audit does not support:**

- The existence of `library_3d_ready.json` by itself proves that all coordinates were individually validated physically or visually.
- The library uses exactly the same sampling strategy or model weights as LUMI; that requires a separate pretraining-configuration review.
- A whole-shard validation split measures generalization to novel Ugi building blocks.
- A dataset-quality PASS proves that HEART predicts mTP, or that adding LNPDB necessarily improves performance.

**Next study task (Priority 2):** characterize LNPDB molecular overlap, test 3D-compatible ingestion, and design a controlled General / Ugi / LNPDB mixed-sampling strategy.
